# Fine-tuning LLMs with LoRA and QLoRA (text-to-SQL)
#### One Flyte 2 pipeline: this MacBook Pro, then a remote NVIDIA GPU.

Adapted from [Union AI's workshop](https://github.com/unionai/workshops/tree/main/tutorials/llm-fine-tuning-lora-qlora). Default model: [SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M) on [b-mc2/sql-create-context](https://huggingface.co/datasets/b-mc2/sql-create-context).

The training recipe is Union’s. **Flyte 2 is why that recipe is the same job on this laptop and on a remote GPU** — and why part 3 can swap in an 8B model without a second stack. Architecture first; then we train.

This talk covers:

- One Flyte 2 pipeline: laptop smoke → remote CUDA (including QLoRA)
- Full vs LoRA vs QLoRA on a small causal LM
- Exact-match SQL: base SmolLM2 vs the fine-tuned model


## One pipeline, two machines

![](images/flyte_pipeline.png)

*Image Credit: original diagram for this tutorial.*

A Hugging Face `Trainer` script is enough to fine-tune SmolLM2 **once, on one box**. It is the wrong shape the moment you have **two machines** and **more than one kind of work**.

This tutorial has both. Follow along on this **16-inch MacBook Pro** (Apple **M4 Max**, **128 GB**, MPS) with a 100-sample smoke test. Union-scale sample counts and **QLoRA** need NVIDIA CUDA / [bitsandbytes](https://github.com/bitsandbytes-foundation/bitsandbytes) 4-bit, so those runs go to a remote **NVIDIA RTX PRO 6000 Blackwell (96 GB)**. Later, part 3 swaps `--model_name` to an 8B causal LM on the same DAG. Serving the fine-tuned model is another task, not another codebase.

**Flyte 2 is the DAG that holds those pieces together.** `workflow.py` is one pipeline:

1. **`prepare_data` (CPU)** — download and format `b-mc2/sql-create-context`. The GPU is not sitting idle on a dataset fetch.
2. **`train` (GPU)** — full, LoRA, or QLoRA. `flyte run --local` on this Mac (`gpu=0`, PyTorch uses MPS). `flyte run` on the remote box (CUDA, including QLoRA). Same task code.
3. **`evaluate` (GPU)** — exact-match SQL vs the untouched base model. Cached: change `--method` without re-downloading the dataset.
4. **`serve` (optional)** — FastAPI in front of the same run.

Without Flyte you maintain a Mac script, a CUDA script, and a third entrypoint when you want to serve or jump to 8B. With Flyte you change the command (`--local` vs remote, `--method`, `--model_name`), not the pipeline. That is the point of this notebook — not “it already imported `flyte` instead of `flytekit`.”

**QLoRA** is CUDA-only. On this Mac set `METHOD` to `lora` or `full`. On the RTX PRO 6000 Blackwell, `qlora` is the same `workflow.py`. The comparison table is **section 3**.


In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/mGalarnyk/Python_Tutorials
    %cd Python_Tutorials/NLP/Fine_Tuning/01_text_to_sql
    !uv pip install -r requirements.txt

from IPython.display import display

from device_utils import describe, detect_device, has_nvidia
from run_grid import collect_completed, format_results, recommend_jobs, run_grid
from run_profile import flyte_local_argv, recommend

ROOT = Path.cwd()
assert (ROOT / "workflow.py").exists(), f"Run this notebook from 01_text_to_sql (cwd={ROOT})"
print("Working directory:", ROOT)

device = detect_device()
print("Device:", describe())
print("NVIDIA / bitsandbytes QLoRA:", has_nvidia())
if not has_nvidia():
    print(
        "QLoRA skipped: bitsandbytes 4-bit needs CUDA. "
        "This machine will run full and LoRA only."
    )

# True = 100-sample LoRA run (laptop / first check).
# False = Union-scale run — use on the remote RTX PRO 6000 Blackwell.
SMOKE = device != "cuda"
# False = one METHOD below. True = full vs LoRA (and QLoRA on CUDA).
GRID = False
profile = recommend(smoke=SMOKE)
print("Jobs if GRID:", recommend_jobs())
profile


## 1. Pipeline parameters

| Flag | Laptop smoke | CUDA default | Meaning |
|------|----------------|--------------|---------|
| `--method` | `lora` | `lora` (`qlora` / `full` optional) | Fine-tuning recipe |
| `--model_name` | `HuggingFaceTB/SmolLM2-135M` | same | Hugging Face model |
| `--epochs` | 1 | 3 | Training epochs |
| `--batch_size` | 2 (MPS) / 4 | 4 | Per-device batch |
| `--max_train_samples` | 100 | 5000 | Train cap |
| `--max_eval_samples` | 20 | 500 | Eval cap |
| `--num_eval_examples` | 10 | 50 | Before/after examples |

Set `METHOD` below to `full`, `lora`, or `qlora`. **QLoRA needs bitsandbytes on NVIDIA CUDA** — it raises a clear error on a Mac. For a full vs LoRA table, set `GRID = True` in the setup cell.

## 2. Train

One method (`METHOD`) or the full vs LoRA grid (`GRID = True`). Same Union `workflow.py` either way.

In [ ]:
METHOD = profile["method"]  # "lora" | "full" | "qlora"

cmd = flyte_local_argv(profile, method=METHOD)
print("GRID:", GRID, " METHOD:", METHOD)
print(" ".join(cmd) if not GRID else "GRID=True → next cell runs full, then LoRA" + (", then QLoRA" if has_nvidia() else " (QLoRA skipped: no CUDA)"))
print("\nThis is Flyte --local (no cluster URL). Skip the next cell if you only wanted the command.")


In [ ]:
# Union's workflow.py, local Flyte. First time downloads SmolLM2 + the SQL dataset.
# GRID=True runs full then LoRA (and QLoRA on CUDA). Resume skips a finished run_<method>.json.
if GRID:
    run_grid(smoke=SMOKE, resume=True)
else:
    result = subprocess.run(cmd, cwd=ROOT)
    sys.exit(result.returncode)


## 3. Results

This section is the tutorial’s comparison table. Metric is **exact-match accuracy** on normalized SQL (Union’s evaluate task): base SmolLM2 vs the fine-tuned model, for **full**, **LoRA**, and **QLoRA** when CUDA is available.

Union’s workshop does not publish a 3-method scoreboard — we run **their pipeline** (same `workflow.py`, same default model `HuggingFaceTB/SmolLM2-135M`) after the Mac/CUDA adaptations, and put the numbers here.

Re-run the next cell anytime. It reads `outputs/run_<method>.json`. This Mac fills **full** and **LoRA** (smoke size unless you set `SMOKE = False` on the remote RTX PRO 6000 Blackwell). **QLoRA is CUDA-only** (bitsandbytes 4-bit).


In [ ]:
runs = collect_completed()
if runs.empty:
    print("No finished runs yet. Run the training cell (one METHOD, or GRID=True), then re-run this cell.")
else:
    print("Exact-match SQL accuracy: base vs fine-tuned (Union pipeline)")
    display(format_results(runs))


## Optional: remote Flyte / serve

Only if you have your own cluster. Do not commit `.flyte/config.yaml` or a hosted GPU URL.

```bash
flyte create config --endpoint YOUR_CLUSTER.hosted.unionai.cloud --project YOUR_PROJECT --domain development --builder remote
flyte run workflow.py pipeline --method lora --epochs 3
python serve.py
```

More: [Flyte run modes](https://www.union.ai/docs/v2/flyte/user-guide/run-modes/) · [Union workshop](https://github.com/unionai/workshops/tree/main/tutorials/llm-fine-tuning-lora-qlora)